In [2]:
from data_generator.data_loader import DataCreator

/home/hice1/stekin6/.conda/envs/llamas/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import argparse


In [4]:
# Running with the same model pool as LLM-TOPLA.
parser = argparse.ArgumentParser()

parser.add_argument("--device", type=str, default="cuda")
parser.add_argument("--model_names", type=str, default="all")
parser.add_argument("--dataset_type", type=str, default="lang", choices=["lang", "vision"])
parser.add_argument("--task_name", type=str, default="musr", choices=["gsm8k", "mmlu_hf", "bbh", "gpqa", "musr"])

# select agent arguments
parser.add_argument("--alpha", type=float, default=1.0, help="size penalty constant")
parser.add_argument("--select_agent_epoch", type=int, default=15)
parser.add_argument("--select_agent_gamma", type=float, default=0.99)
parser.add_argument("--select_agent_clip_epsilon", type=float, default=0.2)
parser.add_argument("--select_agent_lr", type=float, default=0.001)
parser.add_argument("--focal_div_weight", type=float, default=0.5)
parser.add_argument("--plurality_voting_weight", type=float, default=0.5)
parser.add_argument("--fleiss_kappa_weight", type=float, default=0)
parser.add_argument("--q_stats_weight", type=float, default=0)
parser.add_argument("--corr_coef_weight", type=float, default=0)
parser.add_argument("--binary_disagreement_weight", type=float, default=0)
parser.add_argument("--kappa_stats_weight", type=float, default=0)
parser.add_argument("--binary_entropy_weight", type=float, default=0)

# ensemble agent arguments
parser.add_argument("--ensemble_agent_epoch", type=int, default=15)
parser.add_argument("--ens_agent_gamma", type=float, default=0.5)
parser.add_argument("--ens_agent_clip_epsilon", type=float, default=0.2)
parser.add_argument("--ens_agent_lr", type=float, default=0.001)

# common argumentss for both agents
parser.add_argument("--window_size", type=int, default=500)
parser.add_argument("--max_tolerance", type=int, default=150)
parser.add_argument("--update_freq", type=int, default=100)

# Simulate command-line arguments
if 'ipykernel' in sys.modules:
    sys.argv = ['notebook.py']  # Replace with desired arguments
else:
    sys.argv = ['script.py']

# Parse the arguments
args = parser.parse_args()

In [ ]:
import os
from config import DATA_DIR

datacreator = DataCreator(args.task_name, model_names=args.model_names, task_type="lang")
print(args.task_name)
model_scores = {}
for i, (train_data, test_data, num_models, ds_name) in enumerate(datacreator.load()):
    print(ds_name)
    novel_data = test_data[:, :-1]
    novel_label = test_data[:, -1]
    acc = []
    base_model_preds = []
    for i, novel_arr in enumerate(np.split(novel_data, num_models, axis=1)):
        pred = novel_arr.argmax(axis=1)
        base_model_preds.append(pred)
        model_acc = np.mean(novel_label == pred)
        acc.append(model_acc)
    acc = np.array(acc)

    data_path = os.path.join(DATA_DIR, ds_name)
    model_names = datacreator._get_model_names(data_path)
    for i in range(len(acc)):
        if model_names[i] not in model_scores.keys():
            model_scores[model_names[i]] 

musr
murder_mysteries
google__gemma-2-27b: 54.00
microsoft__phi-4: 54.00
deepseek-ai__deepseek-llm-67b-chat: 56.00
Qwen__Qwen2.5-72B: 56.00
meta-llama__Meta-Llama-3-70B: 66.00
mistralai__Mixtral-8x22B-v0.1: 58.00
object_placements
google__gemma-2-27b: 48.08
microsoft__phi-4: 32.69
deepseek-ai__deepseek-llm-67b-chat: 51.92
Qwen__Qwen2.5-72B: 51.92
meta-llama__Meta-Llama-3-70B: 53.85
mistralai__Mixtral-8x22B-v0.1: 48.08
team_allocation
google__gemma-2-27b: 38.00
microsoft__phi-4: 40.00
deepseek-ai__deepseek-llm-67b-chat: 46.00
Qwen__Qwen2.5-72B: 48.00
meta-llama__Meta-Llama-3-70B: 42.00
mistralai__Mixtral-8x22B-v0.1: 38.00


In [18]:
import os
from config import DATA_DIR

novel_data = test_data[:, :-1]
novel_label = test_data[:, -1]
acc = []
base_model_preds = []
for i, novel_arr in enumerate(np.split(novel_data, num_models, axis=1)):
    pred = novel_arr.argmax(axis=1)
    base_model_preds.append(pred)
    model_acc = np.mean(novel_label == pred)
    acc.append(model_acc)
acc = np.array(acc)

data_path = os.path.join(DATA_DIR, "bbh")
model_names = datacreator._get_model_names(data_path)
for i in range(len(acc)):
    print(f"{model_names[i]}: {acc[i] * 100:.2f}")

google__gemma-2-27b: 38.00
microsoft__phi-4: 40.00
deepseek-ai__deepseek-llm-67b-chat: 46.00
Qwen__Qwen2.5-72B: 48.00
meta-llama__Meta-Llama-3-70B: 42.00
mistralai__Mixtral-8x22B-v0.1: 38.00


In [ ]:

data = load_dataset(
    f"HuggingFaceEvalInternal/{model_name}-details-private",
    name=f"{model_name}__leaderboard_bbh_{ds_name}",
    split="latest",
    token=hf_token)

In [9]:
from config import hf_token
from datasets import load_dataset
data = load_dataset(
    "HuggingFaceEvalInternal/mistralai__Mixtral-8x22B-v0.1-details-private",
    name="mistral-community__Mixtral-8x22B-v0.1__leaderboard_arc_challenge",
    split="latest",
    token=hf_token
)


ValueError: BuilderConfig 'mistral-community__Mixtral-8x22B-v0.1__leaderboard_arc_challenge' not found. Available: ['mistralai__Mixtral-8x22B-v0.1__leaderboard_arc_challenge', 'mistralai__Mixtral-8x22B-v0.1__leaderboard_bbh', 'mistralai__Mixtral-8x22B-v0.1__leaderboard_bbh_boolean_expressions', 'mistralai__Mixtral-8x22B-v0.1__leaderboard_bbh_causal_judgement', 'mistralai__Mixtral-8x22B-v0.1__leaderboard_bbh_date_understanding', 'mistralai__Mixtral-8x22B-v0.1__leaderboard_bbh_disambiguation_qa', 'mistralai__Mixtral-8x22B-v0.1__leaderboard_bbh_formal_fallacies', 'mistralai__Mixtral-8x22B-v0.1__leaderboard_bbh_geometric_shapes', 'mistralai__Mixtral-8x22B-v0.1__leaderboard_bbh_hyperbaton', 'mistralai__Mixtral-8x22B-v0.1__leaderboard_bbh_logical_deduction_five_objects', 'mistralai__Mixtral-8x22B-v0.1__leaderboard_bbh_logical_deduction_seven_objects', 'mistralai__Mixtral-8x22B-v0.1__leaderboard_bbh_logical_deduction_three_objects', 'mistralai__Mixtral-8x22B-v0.1__leaderboard_bbh_movie_recomm

In [6]:
from config import hf_token

In [7]:
from datasets import load_dataset
data = load_dataset(
    "open-llm-leaderboard/google__gemma-2-27b-details",
    name="google__gemma-2-27b__leaderboard_bbh_boolean_expressions",
    split="latest",
    token=hf_token
)


Generating latest split: 100%|██████████| 250/250 [00:00<00:00, 37256.21 examples/s]
